# 03 · Sorting, limiting & computed columns

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~25 min**

## Learning objectives
- sort with `ORDER BY` (asc/desc) and take top-N with `LIMIT`/`OFFSET`;
- compute new columns with arithmetic and `ROUND`;
- create categories on the fly with `CASE WHEN`.

Order results, take the top-N, compute new columns, and label rows with `CASE`.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## ⚙️ Setup — run this cell first

This connects the notebook to the example database. It works both on **your own computer** and on **Google Colab** — just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

## 1. ORDER BY

In [ ]:
%%sql
SELECT sample_id, ph FROM samples
WHERE ph IS NOT NULL
ORDER BY ph DESC;

## 2. Top-N: ORDER BY + LIMIT
The 10 largest single observations in the study:

In [ ]:
%%sql
SELECT sample_id, taxon_id, count
FROM abundance
ORDER BY count DESC
LIMIT 10;

## 3. OFFSET — skip rows (e.g. the second page)

In [ ]:
%%sql
SELECT sample_id, ph FROM samples
WHERE ph IS NOT NULL
ORDER BY ph DESC
LIMIT 5 OFFSET 5;

## 4. Computed columns
Convert temperature to Fahrenheit on the fly.

In [ ]:
%%sql
SELECT sample_id, temperature_c,
       ROUND(temperature_c * 9.0 / 5 + 32, 1) AS temperature_f
FROM samples
LIMIT 8;

## 5. CASE — build a category
Label each sample by its pH.

In [ ]:
%%sql
SELECT sample_id, ph,
       CASE WHEN ph < 6.5 THEN 'acidic'
            WHEN ph > 7.5 THEN 'alkaline'
            ELSE 'neutral' END AS reaction
FROM samples
WHERE ph IS NOT NULL;

---
## Exercises

**Exercise.** Show the 5 warmest samples (highest `temperature_c`).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, temperature_c FROM samples
ORDER BY temperature_c DESC
LIMIT 5;
```
</details>

**Exercise.** List sample_id and pH rounded to 1 decimal, for samples with a pH.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, ROUND(ph, 1) AS ph_1dp FROM samples
WHERE ph IS NOT NULL;
```
</details>

**Exercise.** Label each sample `shallow` (depth < 10) or `deep` with CASE.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT sample_id, depth_cm,
       CASE WHEN depth_cm < 10 THEN 'shallow' ELSE 'deep' END AS layer
FROM samples
WHERE depth_cm IS NOT NULL;
```
</details>

### Recap
- `ORDER BY col DESC` sorts; `LIMIT`/`OFFSET` take top-N / pages.
- You can compute columns with arithmetic and `ROUND(x, n)`.
- `CASE WHEN … THEN … ELSE … END` builds categories.
- Next: **04 · Aggregation with GROUP BY**.